In [1]:
import os
from dotenv import load_dotenv

load_dotenv(
    r"C:\Users\guruz\OneDrive\Desktop\Lanchan\.env",
    override=True
)

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("API key not found!")

print("API key loaded successfully")

API key loaded successfully


In [2]:
from langchain_openai import ChatOpenAI

llm_openai = ChatOpenAI(
    model="gpt-5-mini",
    temperature=0
)

In [3]:
## Load Doument
from langchain_community.document_loaders import PyPDFLoader

pdf_files = [
    "./doc/01_Traffic_Rules_and_Road_Safety.pdf",
    "./doc/02_Driving_Licence_Process.pdf",
    "./doc/03_School_Policy.pdf",
    "./doc/04_Driving_School_SOP.pdf",
    "./doc/05_FAQ.pdf"
]

docs = []

for pdf_file in pdf_files:
    loader = PyPDFLoader(pdf_file)
    documents = loader.load()
    for doc in documents:
        doc.metadata={
            "document_name": pdf_file.split("/")[-1].replace(".pdf", ""),
            "source": pdf_file,
            "page": doc.metadata["page"],
            "category": "driving_school",
            "document_type": "knowledge_base"
        }
    docs.extend(documents)

    print(f"Loaded: {pdf_file} | Pages: {len(documents)}")

print("\nTotal pages loaded:", len(docs))

C:\Users\guruz\AppData\Local\Temp\ipykernel_29336\2690678501.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded: ./doc/01_Traffic_Rules_and_Road_Safety.pdf | Pages: 5
Loaded: ./doc/02_Driving_Licence_Process.pdf | Pages: 5
Loaded: ./doc/03_School_Policy.pdf | Pages: 6
Loaded: ./doc/04_Driving_School_SOP.pdf | Pages: 7
Loaded: ./doc/05_FAQ.pdf | Pages: 4

Total pages loaded: 27


Convert in to chunks

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

spliter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)
chunk = spliter.split_documents(docs)


In [5]:
print(type(docs))
print(type(docs[0]))
print(docs[0].metadata)

<class 'list'>
<class 'langchain_core.documents.base.Document'>
{'document_name': '01_Traffic_Rules_and_Road_Safety', 'source': './doc/01_Traffic_Rules_and_Road_Safety.pdf', 'page': 0, 'category': 'driving_school', 'document_type': 'knowledge_base'}


## Create a Embedding Model


In [6]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [7]:
from langchain_community.vectorstores import Chroma
vector_db = Chroma.from_documents(
    embedding=embedding_model,
    documents=chunk,
    persist_directory="./vector/"
)


In [8]:
context = vector_db.similarity_search("til me about driving school")

In [9]:
llm_openai.invoke(f"till me about driving school using this context: {context}").content

'Here’s a concise summary of the driving school information from the documents you gave:\n\nOverview and legal status\n- The school is a licensed driving school (must hold a Form 11 licence from the RTO under Rule 24, Central Motor Vehicles Rules, 1989). The original licence is displayed in the office — you can ask to see it.\n- The school issues a driving school completion certificate in Form 5 (Rule 27(d)) to every student who finishes the course. That certificate is attached to your driving licence application where required.\n\nWho can join / age limits\n- 16 years: motorcycle up to 50 cc (guardian consent required).\n- 18 years: motorcycle with gear or car.\n- Students under 18 must have a parent or guardian sign the enrolment form (Motor Vehicles Act, section 4).\n\nDocuments required for admission\n- Proof of age (original).\n- Proof of address (original).\n- Recent passport-size photographs.\n- A working mobile number and an email address.\n- All documents must show the same na

In [10]:
from langchain_community.tools import tool

def search_driving_school_knowledge(query:str)->str:
     """
     Search the driving school's PDF knowledge base.

     Use this tool to answer questions about:
     traffic rules, road safety, driving licence process,
     school policies, SOP and FAQs.

     Always use this tool before answering the user.
     """

     results = vector_db.similarity_search(query,k=4)

     if not results:
        return "NO_RELEVANT_INFORMATION_FOUND"

     context =[]
     for i,doc in enumerate(results,start=1):
          source = doc.metadata.get("source", "Unknown")
          page = doc.metadata.get("page", "Unknown")
          category = doc.metadata.get("category", "Unknown")
          context.append(
               f"""

               Content:
               {doc.page_content}
               """
          )
     return "\n\n".join(context)

In [11]:
from langchain_core.prompts import ChatPromptTemplate

my_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are a Driving School AI Assistant.

        Your responsibility is to answer questions about the driving school.

        STRICT RULES:

        1. Always use the search_driving_school_knowledge tool
           before answering.

        2. Answer ONLY using information returned by the tool.

        3. Never use your internal knowledge or general knowledge.

        4. If the retrieved information does not contain the answer,
           respond:
           "Sorry, I could not find this information in the
           driving school's knowledge base."

        5. Do not make assumptions or invent information.

        6. Keep answers simple and beginner-friendly.

        7. Mention the document source and page number whenever possible.

        8. If a question is unrelated to driving school,
           politely explain that you can only answer
           driving-school-related questions.
        """
    ),
    (
        "human",
        "{question}"
    )
])

In [12]:
chain = my_prompt | llm_openai

response = chain.invoke({
    "question": "What info can you told...?"
})

print(response.content)

Sorry, I could not find this information in the driving school's knowledge base.


In [14]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm_openai,
    tools=[search_driving_school_knowledge],
    system_prompt="""
    You are a Driving School AI Assistant.

    Always call search_driving_school_knowledge
    before answering.

    Answer only using the retrieved information.

    Never invent answers.

    If information is unavailable, say:
    Sorry, I could not find this information
    in the driving school's knowledge base.

    Mention source and page number whenever possible.
    """
)

response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Can you told me the sefty."
        }
    ]
})

print(response["messages"][-1].content)

Here are the key safety rules from the driving school knowledge base:

- Wear protective equipment
  - Wear a BIS‑certified helmet as rider and pillion on two‑wheelers. (Traffic Rules and Road Safety, Page 3)
  - Driver and passengers should wear seat belts in cars; rear‑seat belt enforcement is active in many cities. (Traffic Rules and Road Safety, Page 3)

- Special rule for young children on motorcycles
  - A child below 4 years must use a safety harness (attached to the rider) and a crash helmet; motorcycle speed must not exceed 40 km/h. (Traffic Rules and Road Safety, Page 3)

- Do not drive distracted or impaired
  - Never use a phone while driving. Never drive after drinking; legal alcohol limit is 30 mg alcohol per 100 ml blood (safest: do not drive at all after drinking). (FAQ, Page 4)

- Basic road behaviour
  - Obey signals and signs, keep to the left, keep a safe distance, and do not drive if you are tired. (FAQ, Page 4)
  - Road markings: continuous centre line = do not cr